In [1]:
from pathlib import Path
from dataclasses import replace
from IPython.display import Markdown, display

PROJECT = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
INPUTS = PROJECT / "inputs"

%cd {PROJECT}

from table2text import Settings, Table2TextWorkflow
from table2text.schemas import AuditMode

/Users/realgobs/Documents/MScproject/table2text_pydanticai


In [2]:
# Check input files
for path in sorted(INPUTS.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT))

inputs/Cas.csv
inputs/MakeModel2016.csv
inputs/Veh.csv
inputs/basketball_data.json
inputs/dftRoadSafety_Accidents_2016.csv
inputs/full_format_recipes.json
inputs/heart_disease_uci.csv
inputs/weatherHistory.csv


In [3]:
import os, getpass
from pydantic_ai import Agent

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DeepSeek API key: ")

agent = Agent("deepseek:deepseek-chat", output_type=str)
result = await agent.run("Reply with exactly: OK")
print(result.output)

OK


In [4]:
from pathlib import Path

from table2text import Settings, Table2TextWorkflow
from table2text.schemas import AuditMode, EvaluationFieldPolicy

project_dir = Path(
    "/Users/realgobs/Documents/MScproject/table2text_pydanticai"
)

settings = Settings(
    use_llm=True,
    structured_output_mode="prompted",
    max_total_tokens=160_000,
    max_agent_requests=8,

    data_understanding_model="deepseek:deepseek-chat",
    orchestrator_model="deepseek:deepseek-chat",
    evidence_model="deepseek:deepseek-chat",
    verifier_model="deepseek:deepseek-chat",
    writer_model="deepseek:deepseek-chat",
    auditor_model="deepseek:deepseek-chat",

    output_dir=project_dir / "runs_notebook",
)

workflow = Table2TextWorkflow(settings)

field_policy = EvaluationFieldPolicy(
    operational_input_paths=[
        "game_id",
        "date",
        "location",
        "overtime",
        "teams",
    ],
    held_out_reference_paths=["summary"],
    metadata_paths=["basketballreference"],
)

result = await workflow.run(
    inputs=[project_dir / "inputs/basketball_data.json"],
    request="Understand the dataset and give a neutral report.",
    audit_mode=AuditMode.INTERNAL,
    evaluation_field_policy=field_policy,
)

print("Run ID:", result.run_id)
print("Release status:", result.release_status.value)
print("Final audit:", result.final_audit.decision.value)
print("Writer mode:", result.raw_writer_output.writer_mode)
print("Verified insights:", len(result.insight_ledger.verified_insights))
print(
    "Insight fallback:",
    result.insight_ledger.fallback_reason or "None",
)
print(
    "Verifier notes:",
    result.fact_ledger.verifier_notes or ["None"],
)
print(
    "Report:",
    project_dir / "runs_notebook" / result.run_id / "final_report.md",
)

Run ID: 20260724T151019Z_ecf6ef31da
Release status: human_review_required
Final audit: pass
Writer mode: llm_writer
Verified insights: 4
Insight fallback: None
Verifier notes: ['All candidates are directly supported by evidence with factual_confidence=1.0 and methodological_strength=1.0.', 'No evidence was excluded; all candidates are eligible for writer use.', 'Prohibited interpretations and required caveats from evidence are preserved in each review.']
Report: /Users/realgobs/Documents/MScproject/table2text_pydanticai/runs_notebook/20260724T151019Z_ecf6ef31da/final_report.md


In [5]:
print(
    "Raw writer mode:",
    result.raw_writer_output.writer_mode,
)
print(
    "Primary-evaluation eligible:",
    result.raw_writer_output
    .eligible_for_primary_evaluation,
)
print(
    "Support entries:",
    len(
        result.raw_writer_output
        .sentence_support
    ),
)
print(
    "Quality revision used:",
    result.quality_revised_writer_output
    is not None,
)

Raw writer mode: llm_writer
Primary-evaluation eligible: True
Support entries: 12
Quality revision used: False


In [6]:
quality = (
    result.final_audit
    .quality_assessment
)

print("Quality status:", quality.status.value)

print("\nQuality findings:")
for finding in quality.findings:
    print("-", finding)

print("\nQuality recommendations:")
for recommendation in quality.recommendations:
    print("-", recommendation)

print("\nMethodological warnings:")
for warning in (
    result.final_audit
    .methodological_warnings
):
    print("-", warning)

print("\nAnnotations:")
for annotation in (
    result.final_audit.annotations
):
    print(
        annotation.severity.value,
        annotation.subtype,
        "=>",
        annotation.explanation,
    )

Quality status: revise

Quality findings:
- The report states a verified insight but does not explain its supported analytical implication.
- Required report component `limitations_next_steps` is not clearly covered.
- The report may substitute an unsupported unit of observation: event.

Quality recommendations:
- Use the verified `why_it_matters` content to explain why the combined findings matter instead of restating them.
- Revise the report to cover the required component using verified facts.
- Use neutral unit wording unless verified facts or deterministic profile support establish a more specific unit.

Methodological warnings:

Annotations:
